In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv('./melb_data.csv')

y = data.Price
X = data.drop(['Price'], axis=1)

X_train_full, X_valid_full, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

categorical_cols = [cname for cname in X_train_full.columns if X_train_full[cname].nunique() < 10 and X_train_full[cname].dtype == "string"]

numerical_cols = [cname for cname in X_train_full.columns if X_train_full[cname].dtype in ['int64', 'float64']]

my_cols = categorical_cols + numerical_cols
X_train = X_train_full[my_cols].copy()
X_valid = X_valid_full[my_cols].copy()

print("Setup Complete!")

Setup Complete!


In [3]:
print(X_train.head())

      Type Method             Regionname  Rooms  Distance  Postcode  Bedroom2  \
12167    u      S  Southern Metropolitan      1       5.0    3182.0       1.0   
6524     h     SA   Western Metropolitan      2       8.0    3016.0       2.0   
8413     h      S   Western Metropolitan      3      12.6    3020.0       3.0   
2919     u     SP  Northern Metropolitan      3      13.0    3046.0       3.0   
6043     h      S   Western Metropolitan      3      13.3    3020.0       3.0   

       Bathroom  Car  Landsize  BuildingArea  YearBuilt  Lattitude  \
12167       1.0  1.0       0.0           NaN     1940.0  -37.85984   
6524        2.0  1.0     193.0           NaN        NaN  -37.85800   
8413        1.0  1.0     555.0           NaN        NaN  -37.79880   
2919        1.0  1.0     265.0           NaN     1995.0  -37.70830   
6043        1.0  2.0     673.0         673.0     1970.0  -37.76230   

       Longtitude  Propertycount  
12167    144.9867        13240.0  
6524     144.9005     

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# 使用均值对数值列进行填充
numerical_transformer = SimpleImputer(strategy='mean')

# 使用众数对字符列填充，然后独热编码
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 使用我们熟悉的随机森林构建模型
model = RandomForestRegressor(n_estimators=100, random_state=0)

# 将训练模型和数据预处理打包到一个管道内
my_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])

# 预处理数据，训练模型
my_pipeline.fit(X_train, y_train)

preds = my_pipeline.predict(X_valid)
score = mean_absolute_error(y_valid, preds)
print('MAE:', score)

MAE: 160814.63047680413
